# 03 — Explainability (SHAP, MTS rules, Ollama)

Demonstracja trzech warstw wyjaśnień dla pojedynczych pacjentów.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.models.base import BaseTriageModel
from src.data.preprocessing import build_feature_groups, split_features
from src.explain.shap_explainer import SHAPTriageExplainer, shap_summary_plot, shap_waterfall_plot
from src.explain.mts_rules import rule_based_triage, check_consistency, explain_rule_decision
from src.explain.ollama_medical import OllamaMedicalExplainer
from src.utils.config import MODELS_DIR, TRAIN_PARQUET, TEST_PARQUET, CLASS_NAMES, CLASS_NAMES_PL

In [ ]:
# Załaduj najnowszy model XGBoost
model_path = sorted(MODELS_DIR.glob('xgboost*.joblib'), key=lambda p: p.stat().st_mtime)[-1]
model = BaseTriageModel.load(model_path)
print(f'Załadowano: {model_path.name}')

In [ ]:
df_train = pd.read_parquet(TRAIN_PARQUET)
df_test = pd.read_parquet(TEST_PARQUET)
groups = build_feature_groups(df_train)
X_train, y_train, _ = split_features(df_train, groups, feature_set='triage_only')
X_test, y_test, _ = split_features(df_test, groups, feature_set='triage_only')
X_train = X_train[model.feature_names]
X_test = X_test[model.feature_names]

## SHAP — globalny

In [ ]:
background = X_train.sample(200, random_state=42)
shap_explainer = SHAPTriageExplainer(model, background_data=background)
shap_explainer.fit()

sample = X_test.sample(500, random_state=42)
shap_values = shap_explainer.explain_dataset(sample)
shap_summary_plot(shap_values, sample, show=True)

## SHAP — per pacjent (waterfall)

In [ ]:
patient = X_test.iloc[[0]]
shap_exp = shap_explainer.explain_patient(patient)

print(f"Predykcja: {shap_exp['predicted_class']}")
print(f"Probabilities: {shap_exp['probabilities']}")
print('\nTop cechy ZA:')
for f in shap_exp['top_features_for'][:5]:
    print(f"  {f['feature']}: {f['shap_value']:+.3f} (wartość={f['patient_value']})")
print('\nTop cechy PRZECIW:')
for f in shap_exp['top_features_against'][:3]:
    print(f"  {f['feature']}: {f['shap_value']:+.3f} (wartość={f['patient_value']})")

shap_waterfall_plot(shap_explainer, patient, show=True)

## MTS rule check

In [ ]:
patient_dict = patient.iloc[0].to_dict()
rule = rule_based_triage(patient_dict)
print(explain_rule_decision(rule))

consistency = check_consistency(
    ml_prediction=shap_exp['predicted_class_idx'],
    rule_prediction=rule['suggested_class_idx'],
)
print('\n', consistency['verdict'])

## Ollama medical reasoning (opcjonalnie)

In [ ]:
ollama = OllamaMedicalExplainer(model_name='mistral')
if ollama.is_available():
    text = ollama.explain(
        patient_data=patient_dict,
        predicted_class=shap_exp['predicted_class'],
        predicted_class_pl=CLASS_NAMES_PL[shap_exp['predicted_class_idx']],
        probabilities=shap_exp['probabilities'],
        shap_explanation=shap_exp,
        rule_check=rule,
    )
    print(text)
else:
    print('Ollama niedostępne. Aby aktywować:')
    print('  ollama serve')
    print('  ollama pull mistral')